In [65]:
import pandas as pd
import numpy as np
import json
import re
import ast
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.decomposition import PCA
from joblib import dump

In [66]:
# --- 1. Load the dataset ---
df = pd.read_csv('../../../data/test-final/FINAL_master.csv')
jd = pd.read_csv('../../../data/test-final/jds_clean.csv')
df = df.merge(jd[['jd_id', 'jd_text_embedding']], on='jd_id', how='left')

In [67]:
# --- 2. Build skill2idx from extracted_skills ---
unique_skills = set()
for skills_str in df['extracted_skills']:
    if pd.notna(skills_str):
        skills = [s.strip().lower() for s in skills_str.split(',')]
        unique_skills.update(skills)
for skills_json in jd['skills_list']:
    if pd.notna(skills_json):
        try:
            skills = json.loads(skills_json)
            unique_skills.update([s.lower() for s in skills])
        except:
            continue
skill2idx = {skill: i for i, skill in enumerate(sorted(unique_skills))}
with open("../saved/skill2idx.json", "w") as f:
    json.dump(skill2idx, f)
print(f"✅ Saved skill2idx.json with {len(skill2idx)} skills")
print(f"Skills: {sorted(unique_skills)}")

✅ Saved skill2idx.json with 33 skills
Skills: ['.net', 'agile', 'api design', 'artificial intelligence', 'azure', 'ci cd devops', 'cloud deployment', 'cloud platforms', 'code patching', 'communication', 'data analysis', 'data engineering', 'data pipelines', 'data preprocessing', 'data visualization', 'deep learning', 'documentation', 'full stack development', 'java', 'javascript', 'machine learning', 'mentorship', 'microsoft technologies', 'nodejs', 'predictive modeling', 'problem solving', 'python', 'react', 'sql', 'statistical analysis', 'system design', 'troubleshooting', 'winforms']


In [68]:
def parse_emb(s):
    try:
        return np.array(json.loads(s))
    except:
        parts = re.split(r'[,\s]+', s.strip().lstrip('[').rstrip(']'))
        return np.array([float(x) for x in parts if x])

def parse_skills_vec(s):
    try:
        return np.array(json.loads(s), dtype=np.float32)
    except:
        print(f"Error parsing skills_vector: {s}")
        return np.zeros(len(skill2idx), dtype=np.float32)

df['text_vec'] = df['transcript_embedding'].apply(parse_emb)
df['jd_vec'] = df['jd_text_embedding'].apply(parse_emb)
df['skills_vec'] = df['skills_vector'].apply(parse_skills_vec)

In [69]:
#Coerce interviewer-rating columns to float, fill NAs with 0
rating_cols = [
    'Ratings.Technical_Proficiency',
    'Ratings.Problem_Solving_Ability',
    'Ratings.Communication_Skills',
    'Ratings.Cultural_Team_Fit',
    'Ratings.Adaptability_Learning'
]
df[rating_cols] = df[rating_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)

# Parse segment_count
df['segment_count'] = pd.to_numeric(df['segment_count'], errors='coerce').fillna(df['segment_count'].mean())

# Concatenate into struct_vec
df['struct_vec'] = df.apply(lambda r: np.concatenate([
    r['skills_vec'],
    [r['segment_count']],
    r[rating_cols].values.astype(float)
]), axis=1)

In [70]:
# --- 4. Filter out bad rows ---
df = df[(df['struct_vec'].map(len) > 0)
        & (df['text_vec'].map(len) > 0)
        & (df['jd_vec'].map(len) > 0)]
print(f"Rows after filtering: {len(df)}")

Rows after filtering: 301


In [71]:
# --- 5. Debug shapes ---
X_struct = np.vstack(df['struct_vec'].values)
X_text = np.vstack(df['text_vec'].values)
X_jd = np.vstack(df['jd_vec'].values)
print("X_struct shape:", X_struct.shape)
print("X_text shape:", X_text.shape)
print("X_jd shape:", X_jd.shape)

X_struct shape: (301, 40)
X_text shape: (301, 384)
X_jd shape: (301, 384)


In [80]:
print(X_jd.shape[1])

384


In [ ]:
jd_dim_expected = 384
text_dim_expected = 384
if X_jd.shape[1] > jd_dim_expected:
    pca_jd = PCA(n_components=jd_dim_expected)
    X_jd = pca_jd.fit_transform(X_jd)
    dump(pca_jd, "../saved/pca_jd.joblib")
    print("✅ Saved pca_jd.joblib")
if X_text.shape[1] > text_dim_expected:
    pca_text = PCA(n_components=text_dim_expected)
    X_text = pca_text.fit_transform(X_text)
    dump(pca_text, "../saved/pca_text.joblib")
    print("✅ Saved pca_text.joblib")

In [58]:
# --- 7. Targets ---
y_reg = df['Overall_Score'].values
rec_cols = ['rec_Hire', 'rec_Consider', 'rec_Reject']
y_cls = df[rec_cols].values.argmax(axis=1)

In [59]:
# --- 8. CV setup ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)
mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

In [60]:
# --- 9. Evaluate unimodal & simple fusion ---
def eval_combo(X, y_reg, y_cls, label):
    reg = GradientBoostingRegressor(random_state=42)
    mse = cross_val_score(reg, X, y_reg, cv=cv, scoring=mse_scorer)
    rmse = np.sqrt(-mse)
    clf = RandomForestClassifier(random_state=42)
    acc = cross_val_score(clf, X, y_cls, cv=cv, scoring='accuracy')
    print(f"-- {label} --")
    print(f"  RMSE: {np.round(rmse, 3)}  Mean: {rmse.mean():.3f}")
    print(f"  Acc : {np.round(acc, 3)}  Mean: {acc.mean():.3f}\n")

In [61]:
eval_combo(X_struct, y_reg, y_cls, "Structured-Only (+ratings)")
eval_combo(X_text, y_reg, y_cls, "Text-Only")
eval_combo(X_jd, y_reg, y_cls, "JD-Only")
eval_combo(np.hstack([X_struct, X_text]), y_reg, y_cls, "Struct+Text")
eval_combo(np.hstack([X_struct, X_jd]), y_reg, y_cls, "Struct+JD")
eval_combo(np.hstack([X_text, X_jd]), y_reg, y_cls, "Text+JD")
eval_combo(np.hstack([X_struct, X_jd, X_text]), y_reg, y_cls, "Full Fusion")

-- Structured-Only (+ratings) --
  RMSE: [0.059 0.078 0.059 0.07  0.056]  Mean: 0.064
  Acc : [0.902 0.95  0.933 0.95  0.933]  Mean: 0.934

-- Text-Only --
  RMSE: [0.106 0.099 0.106 0.088 0.089]  Mean: 0.098
  Acc : [0.705 0.65  0.767 0.767 0.717]  Mean: 0.721

-- JD-Only --
  RMSE: [0.214 0.203 0.23  0.233 0.231]  Mean: 0.222
  Acc : [0.377 0.2   0.3   0.317 0.3  ]  Mean: 0.299

-- Struct+Text --
  RMSE: [0.065 0.073 0.064 0.072 0.07 ]  Mean: 0.069
  Acc : [0.852 0.883 0.883 0.933 0.917]  Mean: 0.894

-- Struct+JD --
  RMSE: [0.065 0.068 0.058 0.065 0.057]  Mean: 0.062
  Acc : [0.869 0.933 0.883 0.933 0.9  ]  Mean: 0.904

-- Text+JD --
  RMSE: [0.106 0.1   0.106 0.092 0.088]  Mean: 0.098
  Acc : [0.738 0.7   0.733 0.7   0.8  ]  Mean: 0.734

-- Full Fusion --
  RMSE: [0.065 0.074 0.06  0.069 0.071]  Mean: 0.068
  Acc : [0.885 0.817 0.867 0.917 0.9  ]  Mean: 0.877



In [62]:
# --- 10. Re-train & save Full-Fusion baselines ---
X_full = np.hstack([X_struct, X_jd, X_text])
print("X_full shape:", X_full.shape)

reg_full = GradientBoostingRegressor(random_state=42).fit(X_full, y_reg)
clf_full = RandomForestClassifier(random_state=42).fit(X_full, y_cls)

dump(reg_full, "../saved/full_regressor.joblib")
dump(clf_full, "../saved/full_classifier.joblib")
print("✅ Retrained & saved full_regressor.joblib & full_classifier.joblib")
print("reg.n_features_in_ =", reg_full.n_features_in_)

X_full shape: (301, 808)
✅ Retrained & saved full_regressor.joblib & full_classifier.joblib
reg.n_features_in_ = 808


In [63]:
import os
import joblib
REG_FILE = "../saved/full_regressor.joblib"
# 1.2) Load the regressor and inspect its expected #features:
reg = joblib.load(REG_FILE)
print("reg.n_features_in_ =", reg.n_features_in_)

reg.n_features_in_ = 808
